# Cleaning and Skill Extraction

This notebook documents the transformation from raw Adzuna India job postings to the clean analysis table.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from clean_jobs import (
    SKILL_PATTERNS,
    clean_postings,
    find_latest_run,
    load_postings,
)

## Locate the bronze data

The collector was initially run from the `scripts` directory, so this notebook checks both possible bronze locations.

In [2]:
bronze_root = PROJECT_ROOT / "data" / "bronze"
bronze_run = find_latest_run(bronze_root)

print(f"Bronze run: {bronze_run}")

Bronze run: d:\data analysis projects\Job market intelligence\data\bronze\20260827T062819Z


## Load raw postings

The raw JSON remains unchanged. The notebook loads all job-search response files and combines their result records.

In [3]:
postings = load_postings(bronze_run)

print(f"Raw postings loaded: {len(postings):,}")

raw_preview = pd.DataFrame(postings)
raw_preview.head()

Raw postings loaded: 1,800


,created,title,salary_is_predicted,company,longitude,__CLASS__,location,category,description,latitude,redirect_url,id,contract_time,adref,_role_category,salary_max,salary_min,contract_type
0,2026-08-12T01:59:43Z,AI Engineer,0,{'__CLASS__': 'Adzuna::API::Response::Company'...,73.84735,Adzuna::API::Response::Job,{'__CLASS__': 'Adzuna::API::Response::Location...,{'__CLASS__': 'Adzuna::API::Response::Category...,"This job is with Capco, an inclusive employer ...",18.50620,https://www.adzuna.in/land/ad/5837139879?se=QE...,5837139879,full_time,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiUUU0MW51Q2g4Ukc4Z...,ai_engineer,NaN,NaN,NaN
1,2026-08-18T18:03:43Z,AI Engineer,0,"{'display_name': 'Morningstar', '__CLASS__': '...",72.84415,Adzuna::API::Response::Job,{'__CLASS__': 'Adzuna::API::Response::Location...,{'__CLASS__': 'Adzuna::API::Response::Category...,"This job is with Morningstar, an inclusive emp...",19.00821,https://www.adzuna.in/land/ad/5846723676?se=QE...,5846723676,full_time,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiUUU0MW51Q2g4Ukc4Z...,ai_engineer,NaN,NaN,NaN
2,2026-08-12T01:59:40Z,AI Engineer_Agentic AI,0,"{'display_name': 'Capco', '__CLASS__': 'Adzuna...",NaN,Adzuna::API::Response::Job,"{'area': ['India'], 'display_name': 'India', '...",{'__CLASS__': 'Adzuna::API::Response::Category...,"This job is with Capco, an inclusive employer ...",NaN,https://www.adzuna.in/land/ad/5837139696?se=QE...,5837139696,full_time,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiUUU0MW51Q2g4Ukc4Z...,ai_engineer,NaN,NaN,NaN
3,2026-08-12T11:59:19Z,Principal AI Engineer,0,"{'display_name': 'Cornerstone OnDemand', '__CL...",78.50806,Adzuna::API::Response::Job,"{'area': ['India', 'Telangana', 'Hyderabad'], ...","{'label': 'IT Jobs', 'tag': 'it-jobs', '__CLAS...","This job is with Cornerstone OnDemand, an incl...",17.40275,https://www.adzuna.in/land/ad/5837843493?se=QE...,5837843493,full_time,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiUUU0MW51Q2g4Ukc4Z...,ai_engineer,NaN,NaN,NaN
4,2026-08-12T01:59:18Z,AI Engineer - India,0,{'__CLASS__': 'Adzuna::API::Response::Company'...,78.50806,Adzuna::API::Response::Job,"{'area': ['India', 'Telangana', 'Hyderabad'], ...","{'tag': 'it-jobs', 'label': 'IT Jobs', '__CLAS...","This job is with Cornerstone OnDemand, an incl...",17.40275,https://www.adzuna.in/land/ad/5837138874?se=QE...,5837138874,full_time,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTgzNzEzODg3NCIsI...,ai_engineer,NaN,NaN,NaN


## Clean and deduplicate

The cleaning function standardizes cities, converts salary values to numeric values, handles missing salaries, and removes duplicate postings.

In [4]:
clean = clean_postings(postings)

print(f"Unique postings: {len(clean):,}")
print(f"Removed duplicates: {len(postings) - len(clean):,}")

clean.head()

Unique postings: 1,285
Removed duplicates: 515


,posting_id,role_category,city,company,salary_min,salary_max,posted_date,sql,python,power_bi,tableau,excel,azure,aws,spark,r,snowflake,dbt,genai_llm
0,5837139879,ai_engineer,Pune,Capco,<NA>,<NA>,2026-08-12 01:59:43+00:00,False,False,False,False,False,False,False,False,False,False,False,True
1,5846723676,ai_engineer,Mumbai,Morningstar,<NA>,<NA>,2026-08-18 18:03:43+00:00,False,False,False,False,False,False,False,False,False,False,False,False
2,5837139696,ai_engineer,India,Capco,<NA>,<NA>,2026-08-12 01:59:40+00:00,False,False,False,False,False,False,False,False,False,False,False,False
3,5837843493,ai_engineer,Hyderabad,Cornerstone OnDemand,<NA>,<NA>,2026-08-12 11:59:19+00:00,False,False,False,False,False,False,False,False,False,False,False,False
4,5837138874,ai_engineer,Hyderabad,Cornerstone OnDemand,<NA>,<NA>,2026-08-12 01:59:18+00:00,False,False,False,False,False,False,False,False,False,False,False,False


## Inspect missing values

In [5]:
clean[["salary_min", "salary_max", "posted_date", "city", "company"]].isna().sum()

salary_min     1071
salary_max     1071
posted_date       0
city              0
company           0
dtype: int64

## Skill extraction

Each skill flag is created by case-insensitive regular-expression matching against the posting description.

In [6]:
skill_rates = (
    clean[list(SKILL_PATTERNS)]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .round(1)
    .rename("mention_rate_percent")
    .to_frame()
)

skill_rates

,mention_rate_percent
python,7.9
genai_llm,7.1
sql,6.4
azure,3.3
spark,2.8
power_bi,2.6
aws,2.6
excel,2.5
r,2.4
tableau,1.8


In [7]:
role_skill_rates = (
    clean.groupby("role_category")[list(SKILL_PATTERNS)]
    .mean()
    .mul(100)
    .round(1)
)

role_skill_rates

,sql,python,power_bi,tableau,excel,azure,aws,spark,r,snowflake,dbt,genai_llm
role_category,,,,,,,,,,,,
ai_engineer,1.7,7.0,0.0,0.2,0.0,3.8,2.9,1.2,1.0,0.0,0.0,18.0
business_analyst,2.5,0.7,1.1,0.0,1.4,0.0,0.0,0.0,3.5,0.0,0.0,0.4
data_analyst,12.0,9.5,10.5,7.6,9.8,1.5,1.1,1.5,2.5,0.4,0.0,2.2
data_engineer,13.1,15.4,0.5,0.5,0.5,9.0,7.7,12.2,3.2,5.0,0.5,1.4
data_scientist,6.7,12.4,0.0,0.0,0.0,3.4,1.1,0.0,3.4,0.0,0.0,6.7


## Export the silver table

The clean output is saved at the project-level `data/silver` directory for SQL and Power BI.

In [8]:
silver_dir = PROJECT_ROOT / "data" / "silver"
silver_dir.mkdir(parents=True, exist_ok=True)

output_path = silver_dir / "jobs_clean.csv"
clean.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%dT%H:%M:%SZ",
)

print(f"Saved: {output_path}")

Saved: d:\data analysis projects\Job market intelligence\data\silver\jobs_clean.csv
